# GF4/E2M1 Network Throughput Benchmark

Measures **single-token decode latency** (ms/tok) on an A100 40 GB across three configs:

| Config | Description |
|---|---|
| `fp16_baseline` | Unquantized FP16 — standard matmul |
| `w4a16_fused` | 4-bit E2M1 weights, fused dequant+GEMV kernel — fp16 W never materialised |
| `w4a4_fused` | Same + Hadamard rotation + GF4 encode/decode on activations |

Speedup **grows with model size** — OPT-125 M may be compute-bound; OPT-6.7 B shows the HBM-bandwidth-bound regime where the 4x traffic reduction dominates.

> **Runtime:** set to A100 (Runtime → Change runtime type → A100 GPU)

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers datasets accelerate scipy huggingface_hub

# ── 2. Clone repo ─────────────────────────────────────────────────────────────
# Update REPO_URL and SUBDIR to match your repository layout.
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'   # <-- EDIT
SUBDIR   = 'Thesis_Compression/Python_Jenks_Test/Jenks_Tests/CUDA_FP4_Test'  # <-- EDIT

import os, subprocess
_repo_name = REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
if not os.path.isdir(_repo_name):
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL], check=True)
os.chdir(os.path.join(_repo_name, SUBDIR))
print('Working directory:', os.getcwd())

In [ ]:
import os, shutil, glob, sys, ctypes
import torch

# ── 1. Wipe every cache location torch might consult ─────────────────────────
for _d in [os.path.expanduser('~/.cache/torch_extensions'),
           os.path.join(os.getcwd(), '_ext_build')]:
    if os.path.isdir(_d):
        shutil.rmtree(_d)
        print(f'Cleared: {_d}')

BUILD_DIR = os.path.join(os.getcwd(), '_ext_build')
os.makedirs(BUILD_DIR, exist_ok=True)

# ── 2. Verify source files ────────────────────────────────────────────────────
SOURCES = [
    'bindings.cpp',
    'hadamard_kernel.cu',
    'gf4_encode_kernel.cu',
    'e2m1_fused_gemv_kernel.cu',
    'gf4_fused_gemv_kernel.cu',
    'hessian_weight_quant_kernel.cu',
]
missing = [s for s in SOURCES if not os.path.exists(s)]
if missing:
    raise FileNotFoundError(f'Missing: {missing}\ncwd: {os.getcwd()}')

print(f'torch {torch.__version__}  |  Python {sys.version.split()[0]}  |'
      f'  CUDA {torch.version.cuda}')
print('Sources OK. Building from:', os.getcwd())

# ── 3. Compile — build_directory passed directly (bypasses env-var caching) ──
from torch.utils.cpp_extension import load

try:
    ext = load(
        name='gf4_kernels',
        sources=SOURCES,
        extra_cflags=['-O3'],
        extra_cuda_cflags=['-O3', '--use_fast_math'],
        build_directory=BUILD_DIR,   # ← direct param, not env var
        verbose=True,
    )
    print('\nCUDA extension ready.')

    # Smoke test
    _x = torch.randn(32, device='cuda')
    _s = torch.ones(32, device='cuda')
    ext.hadamard_fwht(_x.unsqueeze(0), 32, _s)
    print('Smoke test passed.')

except Exception as _e:
    # ── Diagnose: show exactly what torch wrote (or didn't write) ────────────
    print(f'\nBUILD FAILED: {_e}\n')
    _bd = os.path.join(BUILD_DIR, 'gf4_kernels')
    if not os.path.isdir(_bd):
        print('Build sub-directory was never created.')
        print('→ torch did not invoke ninja at all.')
        print('  Possible cause: torch/Python version mismatch.')
        print(f'  Try: !pip install --upgrade torch  (current: {torch.__version__})')
    else:
        print(f'Contents of {_bd}:')
        for _f in sorted(os.listdir(_bd)):
            _sz = os.path.getsize(os.path.join(_bd, _f))
            print(f'  {_f:<45} {_sz:>8} bytes')

        # Print ninja log — contains the actual compiler error
        for _log in ['.ninja_log', 'build.ninja']:
            _lp = os.path.join(_bd, _log)
            if os.path.exists(_lp):
                print(f'\n--- {_log} (last 3000 chars) ---')
                with open(_lp) as _lf:
                    print(_lf.read()[-3000:])

        # Test any .so that exists (might have wrong version hash)
        _sos = sorted(glob.glob(os.path.join(_bd, '*.so')))
        if _sos:
            for _so in _sos:
                print(f'\nctypes test on {os.path.basename(_so)}:')
                try:
                    ctypes.CDLL(_so)
                    print('  ctypes.CDLL OK → .so is valid, issue is in torch importer')
                    print('  Try: import importlib.util; spec = importlib.util.spec_from_file_location'
                          f'("gf4_kernels", "{_so}")')
                except OSError as _ce:
                    print(f'  ctypes.CDLL FAILED: {_ce}')
                    print('  → shared library dependency missing (ldd output below)')
                    import subprocess
                    _ldd = subprocess.run(['ldd', _so], capture_output=True, text=True)
                    print(_ldd.stdout)
        else:
            print('\nNo .so files found in build dir → nvcc/ninja did not produce output.')
            print('Paste the verbose output above (or the build.ninja contents) for diagnosis.')

    raise

In [ ]:
# ── 4. Configuration — edit here ─────────────────────────────────────────────

MODELS = [
    'facebook/opt-125m',
    'facebook/opt-1.3b',
    # 'facebook/opt-2.7b',
    # 'facebook/opt-6.7b',
    # 'facebook/opt-13b',
    # 'mistralai/Mistral-7B-v0.1',
    # 'Qwen/Qwen2.5-7B',
    # 'Qwen/Qwen2.5-14B',
]
HF_TOKEN        = ''    # 'hf_...' for gated models

# Purge each model's HF disk cache after its run so sequential models don't
# accumulate on Colab's ~78 GB disk. OPT-13B is ~26 GB; Qwen-14B is ~28 GB.
# Set False only if you want to re-run the same model without re-downloading.
PURGE_HF_CACHE  = True

# Batch sizes to sweep. At batch=1 the fused GEMV kernel runs; at batch>1
# the forward falls back to dense fp16 matmul (no fused batched kernel exists),
# so W4A16 reverts to fp16-equivalent cost. The crossover where W4A4 activation
# encoding amortises typically appears around batch 8-16 for 7B+ hidden dims.
BATCH_SIZES     = [1, 2, 4, 8, 16, 32, 64]

N_DECODE_ITERS  = 200   # timed single-token iterations per (config, batch_size)
N_WARMUP        = 30    # untimed warmup iterations
N_CALIB_WINDOWS = 4     # WikiText-2 calibration windows
SEQLEN          = 512   # calibration window length (tokens)

# !! MUST match E2M1_BLOCK in e2m1_fused_gemv_kernel.cu (hardcoded as 16).
HESS_BLOCK      = 16

SKIP_W4A4       = False
OUT_CSV         = 'throughput_results.csv'

In [ ]:
# ── 5. Imports and constants ─────────────────────────────────────────────────
import os, sys, time, zlib
import numpy as np
import torch
import torch.nn as nn

# Synchronous kernel launches: errors are reported at the right call site
# instead of propagating asynchronously to an unrelated op.
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

# ── GPU context health check ──────────────────────────────────────────────────
# If the previous run crashed with an illegal memory access, the CUDA context
# is poisoned. Detect that here with a clear message instead of a cryptic error
# three cells later.
try:
    _chk = torch.zeros(1, device='cuda') + 1
    assert _chk.item() == 1.0
    del _chk
except Exception as _ctx_err:
    raise RuntimeError(
        'CUDA context is in a broken state (likely from a previous bad kernel).\n'
        'Fix: Runtime -> Restart runtime, then run all cells again.\n'
        'Underlying error: ' + str(_ctx_err))

DEVICE        = torch.device('cuda')
KAPPA         = 100.0
POWER_ITERS   = 30
GF4_BLOCK     = 32
CLIP_RATIO    = 2.5
ACT_HAD_BLOCK = 32
RETAIN_FP16   = ('fc2', 'down_proj', 'lm_head')

E2M1_CB = torch.tensor([
    [1.0, 1.5,   2.0, 3.0, 4.0, 6.0, 8.0,  12.0],
    [0.5, 0.75,  1.0, 1.5, 2.0, 3.0, 4.0,   6.0],
    [0.25, 0.375, 0.5, 0.75, 1.0, 1.5, 2.0,  3.0],
], dtype=torch.float32, device=DEVICE)

_L2_BUF = torch.empty(128 * 1024 * 1024 // 4, dtype=torch.float32, device=DEVICE)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

_blk = os.environ['CUDA_LAUNCH_BLOCKING']
print('Imports OK. CUDA device:', torch.cuda.get_device_name(0))
print('CUDA_LAUNCH_BLOCKING=' + _blk + '  (synchronous errors enabled)')

In [ ]:
# ── 6. E2M1 unpack + dequantize ──────────────────────────────────────────────

def _unpack_codes(packed):
    lo  = (packed & 0xF).to(torch.uint8)
    hi  = ((packed >> 4) & 0xF).to(torch.uint8)
    out = torch.empty(packed.shape[0], packed.shape[1] * 2,
                      dtype=torch.uint8, device=packed.device)
    out[:, 0::2] = lo
    out[:, 1::2] = hi
    return out


def dequantize_e2m1(W_codes, W_alpha, W_bias, block_size):
    M        = W_codes.shape[0]
    codes_u8 = _unpack_codes(W_codes)
    n_blk    = codes_u8.shape[1] // block_size
    codes_u8 = codes_u8.view(M, n_blk, block_size)
    sign     = torch.where((codes_u8 & 0x8) != 0, -1.0, 1.0)
    mag_tbl  = E2M1_CB[W_bias.long()]
    idx      = (codes_u8 & 0x7).long()
    del codes_u8
    mag = torch.gather(mag_tbl, 2, idx)
    del idx, mag_tbl
    W_hat = sign
    W_hat.mul_(mag).mul_(W_alpha.float().unsqueeze(-1))
    return W_hat.reshape(M, n_blk * block_size)

In [ ]:
# ── 7. Fused forward constructors ────────────────────────────────────────────

# opt_pop2 lattice codebook levels: {0,2,4,6,8,10,12,16}/16
# Matches gf4_fused_lattice_gemv_kernel's lattice_decode_one exactly.
LATTICE_LVL_GPU = torch.tensor(
    [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 1.0],
    dtype=torch.float32, device=DEVICE)
GF4_WBLK = 32   # block size for GF4/lattice weight kernels (matches GF4_BLOCK in gf4_common.cuh)


def make_fused_w4a16_forward(packed_cpu, alpha_cpu, bias_cpu,
                              block_size, K, orig_bias_cpu):
    packed_d   = packed_cpu.to(DEVICE)
    alpha_d    = alpha_cpu.to(DEVICE)
    bias_arr_d = bias_cpu.to(DEVICE)
    W_fp16     = dequantize_e2m1(packed_d, alpha_d, bias_arr_d, block_size).half()
    orig_d     = orig_bias_cpu.to(DEVICE) if orig_bias_cpu is not None else None

    def forward(x):
        orig_shape = x.shape
        x_2d = x.reshape(-1, K)
        if x_2d.shape[0] == 1:
            y = ext.e2m1_fused_gemv(
                packed_d, alpha_d, bias_arr_d,
                x_2d.squeeze(0).to(DEVICE).half(), K)
        else:
            y = x_2d.to(DEVICE).half() @ W_fp16.t()
        if orig_d is not None:
            y = y + orig_d
        return y.to(x.dtype).reshape(*orig_shape[:-1], y.shape[-1])
    return forward


def make_fused_lattice16_forward(lat_codes_cpu, lat_alpha_cpu, K, orig_bias_cpu):
    """
    W_lattice16: weights packed as GF4 codes but encoded with the opt_pop2
    lattice codebook {0,2,4,6,8,10,12,16}/16 and decoded via shift-and-add
    (no 8-entry LUT). Otherwise identical structure to W4A16 — same fused
    GEMV, same per-block fp16 scale, same batch>1 fallback.
    """
    lat_codes_d = lat_codes_cpu.to(DEVICE)
    lat_alpha_d = lat_alpha_cpu.to(DEVICE)

    # Pre-dequantize for the batch>1 dense-matmul fallback
    M = lat_codes_d.shape[0]
    lo       = (lat_codes_d & 0xF).long()
    hi       = ((lat_codes_d >> 4) & 0xF).long()
    codes_u8 = torch.stack([lo, hi], dim=-1).reshape(M, K)
    mag_idx  = codes_u8 & 0x7
    sign     = torch.where((codes_u8 & 0x8) != 0, -1.0, 1.0)
    lat_mag  = LATTICE_LVL_GPU[mag_idx]
    n_blk    = K // GF4_WBLK
    alpha_ex = lat_alpha_d.float().unsqueeze(-1).expand(M, n_blk, GF4_WBLK).reshape(M, K)
    W_fp16   = (sign * lat_mag * alpha_ex).half().contiguous()
    del lo, hi, codes_u8, mag_idx, sign, lat_mag, alpha_ex

    orig_d = orig_bias_cpu.to(DEVICE) if orig_bias_cpu is not None else None

    def forward(x):
        orig_shape = x.shape
        x_2d = x.reshape(-1, K)
        if x_2d.shape[0] == 1:
            y = ext.gf4_lattice_gemv(
                lat_codes_d, lat_alpha_d,
                x_2d.squeeze(0).to(DEVICE).half(), K)
        else:
            y = x_2d.to(DEVICE).half() @ W_fp16.t()
        if orig_d is not None:
            y = y + orig_d
        return y.to(x.dtype).reshape(*orig_shape[:-1], y.shape[-1])
    return forward


def make_fused_w4a4_forward(packed_cpu, alpha_cpu, bias_cpu,
                             block_size, K,
                             d_sign_cpu, mu_cpu, bias_correction_cpu,
                             orig_bias_cpu):
    packed_d   = packed_cpu.to(DEVICE)
    alpha_d    = alpha_cpu.to(DEVICE)
    bias_arr_d = bias_cpu.to(DEVICE)
    W_had_fp16 = dequantize_e2m1(packed_d, alpha_d, bias_arr_d, block_size).half()
    d_sign_d   = d_sign_cpu.to(DEVICE)
    mu_d       = mu_cpu.to(DEVICE)
    bc_d       = bias_correction_cpu.to(DEVICE)
    orig_d     = orig_bias_cpu.to(DEVICE) if orig_bias_cpu is not None else None

    def forward(x):
        orig_shape = x.shape
        x_2d   = x.reshape(-1, K).float().to(DEVICE).contiguous()
        x_had  = ext.hadamard_fwht(x_2d, ACT_HAD_BLOCK, d_sign_d)
        n_blks = x_had.numel() // GF4_BLOCK
        codes, scales = ext.gf4_encode(
            x_had.reshape(-1).contiguous(), CLIP_RATIO, True, mu_d)
        x_q = ext.gf4_decode(codes, scales, n_blks).reshape(x_had.shape)
        if x_2d.shape[0] == 1:
            y = ext.e2m1_fused_gemv(
                packed_d, alpha_d, bias_arr_d,
                x_q.squeeze(0).half(), K).float() + bc_d
        else:
            y = (x_q.half() @ W_had_fp16.t()).float() + bc_d.unsqueeze(0)
        if orig_d is not None:
            y = y + orig_d
        return y.to(x.dtype).reshape(*orig_shape[:-1], y.shape[-1])
    return forward

In [ ]:
# ── 8. Timing utility ────────────────────────────────────────────────────────

@torch.no_grad()
def measure_decode_latency_ms(model, n_iters, n_warmup, batch_size=1):
    """
    Mean single-token decode latency (ms) at a given batch size.
    Input is [batch_size, 1] — one new token per sequence in the batch.
    L2 cache is flushed before every timed iteration for HBM-bound numbers.
    """
    tok_id = torch.full((batch_size, 1), 2, dtype=torch.long, device=DEVICE)

    for _ in range(n_warmup):
        model(tok_id)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)
    total = 0.0
    for i in range(n_warmup + n_iters):
        _L2_BUF.zero_()
        torch.cuda.synchronize()
        start.record()
        model(tok_id)
        end.record()
        torch.cuda.synchronize()
        if i >= n_warmup:
            total += start.elapsed_time(end)
    return total / n_iters

In [ ]:
# ── 9. benchmark_model() ─────────────────────────────────────────────────────

def _purge_hf_cache(model_name):
    """Delete one model's HF hub cache from disk (frees ~7–28 GB per model)."""
    import shutil
    try:
        from huggingface_hub.constants import HF_HUB_CACHE
        base = HF_HUB_CACHE
    except Exception:
        base = os.path.expanduser('~/.cache/huggingface/hub')
    d = os.path.join(base, 'models--' + model_name.replace('/', '--'))
    if os.path.isdir(d):
        shutil.rmtree(d, ignore_errors=True)
        print(f'  [HF cache purged: {d}]')
    else:
        print(f'  [HF cache not found at {d} — already clean]')


def benchmark_model(model_name):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from datasets import load_dataset

    print(f'\n{"="*72}')
    print(f'MODEL: {model_name}')
    print(f'{"="*72}')

    tok   = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16).to(DEVICE).eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e9
    vram_gb  = torch.cuda.memory_allocated() / 1e9
    print(f'Parameters: {n_params:.3f}B  |  GPU VRAM used: {vram_gb:.2f} GB')

    targets = [
        (n, m) for n, m in model.named_modules()
        if isinstance(m, nn.Linear)
        and not any(s in n for s in RETAIN_FP16)
        and m.in_features % HESS_BLOCK == 0
        and m.in_features % GF4_WBLK == 0
    ]
    print(f'Quantizing {len(targets)} linear layers '
          f'(retaining FP16 for {RETAIN_FP16})...')

    # Calibration activations
    _last_err = None
    for repo in ('Salesforce/wikitext', 'wikitext'):
        try:
            ds = load_dataset(repo, 'wikitext-2-raw-v1', split='train')
            break
        except Exception as e:
            _last_err = e
    else:
        raise RuntimeError(f'Could not load wikitext-2: {_last_err}')

    calib_text = '\n\n'.join(ds['text'][:2000])
    calib_ids  = tok(calib_text, return_tensors='pt').input_ids[0]
    n_win      = min(N_CALIB_WINDOWS, calib_ids.numel() // SEQLEN)
    calib_wins = [calib_ids[i*SEQLEN:(i+1)*SEQLEN].unsqueeze(0).to(DEVICE)
                  for i in range(n_win)]

    captured = {n: [] for n, _ in targets}
    hooks    = []
    def _make_hook(name):
        def _h(mod, inp, _out):
            captured[name].append(
                inp[0].detach().reshape(-1, inp[0].shape[-1]).half().cpu())
        return _h
    for n, m in targets:
        hooks.append(m.register_forward_hook(_make_hook(n)))
    with torch.no_grad():
        for w in calib_wins:
            model(w)
    for h in hooks:
        h.remove()
    print(f'Calibration activations captured ({n_win} windows x {SEQLEN} tokens).')

    raw_state = {}
    lat_state = {}
    rot_state = {}
    t_q0 = time.time()

    for name, module in targets:
        K = module.in_features
        X = torch.cat(captured[name], dim=0).to(DEVICE).float().contiguous()
        del captured[name]
        W = module.weight.data.float().to(DEVICE).contiguous()
        M_w = W.shape[0]
        orig_bias = (module.bias.data.float().cpu().clone()
                     if module.bias is not None else None)

        # ── E2M1 Hessian-weighted (feeds W4A16) ──────────────────────────────
        H  = ext.hessian_accumulate(X, HESS_BLOCK)
        Hd = ext.hessian_damp_blocks(H, KAPPA, POWER_ITERS)
        codes, alpha, bias_arr = ext.hessian_weight_solve(W, Hd)
        raw_state[name] = dict(codes=codes.cpu(), alpha=alpha.cpu(),
                               bias_arr=bias_arr.cpu(), K=K, orig_bias=orig_bias)
        del H, Hd, codes, alpha, bias_arr

        # ── Lattice nearest-neighbour (feeds W_lattice16) ─────────────────────
        n_blk_l = K // GF4_WBLK
        W_blk_l = W.reshape(M_w, n_blk_l, GF4_WBLK)
        rms_l   = W_blk_l.pow(2).mean(-1, keepdim=True).sqrt().clamp(1e-8)
        Wn_l    = (W_blk_l / (rms_l * CLIP_RATIO)).clamp(-1.0, 1.0)
        dists_l = (Wn_l.abs().unsqueeze(-1) - LATTICE_LVL_GPU.view(1, 1, 1, -1)).abs()
        idx_l   = dists_l.argmin(-1).to(torch.uint8)
        sign_l  = (Wn_l < 0).to(torch.uint8) << 3
        codes_l = (idx_l | sign_l).reshape(M_w, K)
        packed_l = (codes_l[:, 0::2] | (codes_l[:, 1::2] << 4)).contiguous()
        alpha_l  = (rms_l.squeeze(-1) * CLIP_RATIO).half()
        lat_state[name] = dict(codes=packed_l.cpu(), alpha=alpha_l.cpu(),
                               K=K, orig_bias=orig_bias)
        del W_blk_l, rms_l, Wn_l, dists_l, idx_l, sign_l, codes_l, packed_l, alpha_l

        # ── Hadamard-rotated E2M1 (feeds W4A4) ───────────────────────────────
        seed   = zlib.crc32(name.encode('utf-8')) % (2 ** 31)
        gen    = torch.Generator(device='cpu').manual_seed(seed)
        d_sign = (torch.randint(0, 2, (K,), generator=gen).float() * 2 - 1).to(DEVICE)
        X_had  = ext.hadamard_fwht(X, ACT_HAD_BLOCK, d_sign)
        W_had  = ext.hadamard_fwht(W, ACT_HAD_BLOCK, d_sign)
        H_r    = ext.hessian_accumulate(X_had, HESS_BLOCK)
        Hd_r   = ext.hessian_damp_blocks(H_r, KAPPA, POWER_ITERS)
        c_r, a_r, b_r = ext.hessian_weight_solve(W_had, Hd_r)
        W_hat  = dequantize_e2m1(c_r, a_r, b_r, HESS_BLOCK)
        mu     = X_had.mean(dim=0).contiguous()
        bc     = (W_hat @ mu).contiguous()
        rot_state[name] = dict(codes=c_r.cpu(), alpha=a_r.cpu(), bias_arr=b_r.cpu(),
                               K=K, d_sign=d_sign.cpu(), mu=mu.cpu(),
                               bias_correction=bc.cpu(), orig_bias=orig_bias)
        del X, W, X_had, W_had, H_r, Hd_r, c_r, a_r, b_r, W_hat, mu, bc, d_sign
        torch.cuda.empty_cache()

    print(f'Quantization done in {time.time()-t_q0:.1f} s.')

    orig_fwd = {name: module.forward for name, module in targets}

    # ── FP16 baseline ─────────────────────────────────────────────────────────
    print('\nTiming FP16 baseline...')
    fp16_times = {}
    for bs in BATCH_SIZES:
        fp16_times[bs] = measure_decode_latency_ms(model, N_DECODE_ITERS, N_WARMUP, batch_size=bs)
        print(f'  bs={bs:>2}: {fp16_times[bs]:.2f} ms')

    # ── W4A16 (E2M1 fused GEMV) ───────────────────────────────────────────────
    print('\nTiming W4A16 (E2M1 fused GEMV)...')
    for name, module in targets:
        st = raw_state[name]
        module.forward = make_fused_w4a16_forward(
            st['codes'], st['alpha'], st['bias_arr'], HESS_BLOCK, st['K'], st['orig_bias'])
    w4a16_times = {}
    for bs in BATCH_SIZES:
        w4a16_times[bs] = measure_decode_latency_ms(model, N_DECODE_ITERS, N_WARMUP, batch_size=bs)
        print(f'  bs={bs:>2}: {w4a16_times[bs]:.2f} ms  ({fp16_times[bs]/w4a16_times[bs]:.2f}x)')
    for name, module in targets:
        module.forward = orig_fwd[name]

    # ── W_lattice16 (opt_pop2 lattice fused GEMV, shift-and-add decode) ───────
    print('\nTiming W_lattice16 (lattice fused GEMV)...')
    for name, module in targets:
        st = lat_state[name]
        module.forward = make_fused_lattice16_forward(
            st['codes'], st['alpha'], st['K'], st['orig_bias'])
    lat16_times = {}
    for bs in BATCH_SIZES:
        lat16_times[bs] = measure_decode_latency_ms(model, N_DECODE_ITERS, N_WARMUP, batch_size=bs)
        print(f'  bs={bs:>2}: {lat16_times[bs]:.2f} ms  ({fp16_times[bs]/lat16_times[bs]:.2f}x)')
    for name, module in targets:
        module.forward = orig_fwd[name]

    # ── W4A4 (Hadamard + GF4 activations + E2M1 weights) ─────────────────────
    w4a4_times = {}
    if not SKIP_W4A4:
        print('\nTiming W4A4 (Hadamard + GF4 activations)...')
        for name, module in targets:
            st = rot_state[name]
            module.forward = make_fused_w4a4_forward(
                st['codes'], st['alpha'], st['bias_arr'], HESS_BLOCK, st['K'],
                st['d_sign'], st['mu'], st['bias_correction'], st['orig_bias'])
        for bs in BATCH_SIZES:
            w4a4_times[bs] = measure_decode_latency_ms(model, N_DECODE_ITERS, N_WARMUP, batch_size=bs)
            print(f'  bs={bs:>2}: {w4a4_times[bs]:.2f} ms  ({fp16_times[bs]/w4a4_times[bs]:.2f}x)')
        for name, module in targets:
            module.forward = orig_fwd[name]

    # ── Cleanup: GPU memory + HF disk cache ───────────────────────────────────
    del model
    torch.cuda.empty_cache()
    if PURGE_HF_CACHE:
        _purge_hf_cache(model_name)

    batch_results = [
        dict(batch_size=bs,
             fp16=fp16_times[bs],
             w4a16=w4a16_times[bs],
             lat16=lat16_times[bs],
             w4a4=w4a4_times.get(bs))
        for bs in BATCH_SIZES
    ]
    return dict(model=model_name, n_params=n_params, batches=batch_results)

In [ ]:
# ── 10. Run benchmark ────────────────────────────────────────────────────────
import traceback

all_results = []
for mname in MODELS:
    try:
        r = benchmark_model(mname)
        all_results.append(r)
    except Exception as exc:
        traceback.print_exc()
        all_results.append(dict(model=mname, n_params=float('nan'),
                                fp16=float('nan'), w4a16=float('nan'),
                                w4a4=None))

In [ ]:
# ── 11. Summary table + CSV ──────────────────────────────────────────────────
import csv

def _spd(t_ref, t):
    try:
        return t_ref / t
    except (TypeError, ZeroDivisionError):
        return float('nan')

for r in all_results:
    short = r['model'].split('/')[-1]
    if not r.get('batches'):
        print(f'{short}: (error — see traceback above)')
        continue

    print(f'\n{"="*90}')
    print(f'MODEL: {short}  ({r["n_params"]:.3f}B params)  —  batch size sweep')
    print(f'{"="*90}')
    print(f'{"Batch":>6}  {"FP16 ms":>9}  {"W4A16 ms":>9}  {"W4A16 spd":>10}'
          f'  {"Lat16 ms":>9}  {"Lat16 spd":>10}'
          + (f'  {"W4A4 ms":>9}  {"W4A4 spd":>10}' if not SKIP_W4A4 else ''))
    print('-' * (80 if SKIP_W4A4 else 100))

    for b in r['batches']:
        bs   = b['batch_size']
        fp16 = b['fp16'];  w16 = b['w4a16']; l16 = b['lat16']; w44 = b['w4a4']
        row  = (f'{bs:>6}  {fp16:>9.2f}  {w16:>9.2f}  {_spd(fp16,w16):>9.2f}x'
                f'  {l16:>9.2f}  {_spd(fp16,l16):>9.2f}x')
        if not SKIP_W4A4:
            row += (f'  {w44:>9.2f}  {_spd(fp16,w44):>9.2f}x'
                    if w44 is not None else f'  {"n/a":>9}  {"n/a":>10}')
        print(row)

# Append to CSV
new_file = not os.path.exists(OUT_CSV)
with open(OUT_CSV, 'a', newline='') as f:
    w = csv.writer(f)
    if new_file:
        w.writerow(['timestamp', 'model', 'n_params_B', 'hess_block',
                    'n_calib_windows', 'seqlen', 'n_decode_iters', 'batch_size',
                    'fp16_ms', 'w4a16_ms', 'w4a16_speedup',
                    'lat16_ms', 'lat16_speedup',
                    'w4a4_ms', 'w4a4_speedup'])
    for r in all_results:
        if not r.get('batches'):
            continue
        for b in r['batches']:
            fp16 = b['fp16']; w16 = b['w4a16']; l16 = b['lat16']; w44 = b['w4a4']
            try:
                w.writerow([
                    time.strftime('%Y-%m-%d %H:%M:%S'),
                    r['model'], f"{r['n_params']:.3f}", HESS_BLOCK,
                    N_CALIB_WINDOWS, SEQLEN, N_DECODE_ITERS, b['batch_size'],
                    f'{fp16:.4f}', f'{w16:.4f}', f'{_spd(fp16,w16):.4f}',
                    f'{l16:.4f}', f'{_spd(fp16,l16):.4f}',
                    f'{w44:.4f}' if w44 else 'n/a',
                    f'{_spd(fp16,w44):.4f}' if w44 else 'n/a',
                ])
            except (TypeError, ZeroDivisionError):
                pass

print(f'\nResults appended to {OUT_CSV}')

## Notes

### Batch-size sweep interpretation

| Regime | W4A16 vs FP16 | W4A4 vs FP16 |
|---|---|---|
| batch=1 (decode) | **1.4–1.7×** — fused GEMV avoids materialising fp16 W | often <1× — Hadamard+encode overhead dominates |
| batch>1 (prefill/batched decode) | ≈1× — falls back to dense fp16 matmul | improves as encoding cost amortises |

- **W4A16 at batch>1 reverts to fp16 cost**: the fused GEMV only runs at batch=1; larger batches fall back to a pre-dequantized FP16 matrix, so the HBM savings disappear. A batched fused GEMM kernel would be needed to recover them.
- **W4A4 crossover**: the batch size where W4A4 first beats FP16 shows the amortisation point. For 7B+ hidden dims (d_model=4096) expect crossover around batch 8–16; for OPT-125M/1.3B (d_model=768/2048) it may not appear within the sweep range.
- These numbers use **cold-L2 flushed timing** — realistic for token streaming, not batch throughput.

### H_{m,r} lattice decode (not yet wired in)
The `opt (lattice)` column from the RTX 4080 table (2.62–3.14× vs dense fp16) would come from replacing the GF4 LUT decode in the GEMV kernel with `k * (1/2^m)` arithmetic using compile-time `K_TABLE` constants. That eliminates all shared/constant-memory pressure from decode. Wiring it in requires: (1) arithmetic decode in the kernel, (2) lattice-code encoding at quantization time using the `opt_pop2` codebook from `codebook_suite.py`.

### Llama models
Set `HF_TOKEN = 'hf_...'` in the config cell, then add e.g. `meta-llama/Llama-3.2-1B` or `meta-llama/Llama-2-7b-hf` to `MODELS`. The large hidden dims (d_model=4096) are where both W4A16 fused and the H_{m,r} speedup numbers are strongest.